In [3]:
import gzip
import json
import os
import sys
from pathlib import Path

import orjson
import pandas as pd

In [4]:
DATASET_PATH = r"E:\Coding\Projects\redrob-ranker\dataset\candidates.jsonl"
DATASET_PATH

'E:\\Coding\\Projects\\redrob-ranker\\dataset\\candidates.jsonl'

In [5]:
ARTIFACTS_DIR = Path(r"E:\Coding\Projects\redrob-ranker\dataset\artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR

WindowsPath('E:/Coding/Projects/redrob-ranker/dataset/artifacts')

In [6]:
def open_input(path: str):
    """Open plain or gzipped JSONL transparently."""
    if path.endswith(".gz"):
        return gzip.open(path, "rb")
    return open(path, "rb")


def safe_get(d: dict, *keys, default=None):
    """Nested safe get: safe_get(d, 'a', 'b') == d.get('a', {}).get('b')."""
    for k in keys:
        if not isinstance(d, dict):
            return default
        d = d.get(k, default)
        if d is None:
            return default
    return d


# ---------------------------------------------------------------------------
# Row extractors
# ---------------------------------------------------------------------------

def extract_base(cid: str, rec: dict) -> dict:
    """One row per candidate — static profile fields."""
    p = rec.get("profile", {})
    return {
        "candidate_id":        cid,
        "anonymized_name":     p.get("anonymized_name"),
        "headline":            p.get("headline"),
        "summary":             p.get("summary"),
        "location":            p.get("location"),
        "country":             p.get("country"),
        "years_of_experience": p.get("years_of_experience"),
        "current_title":       p.get("current_title"),
        "current_company":     p.get("current_company"),
        "current_company_size":p.get("current_company_size"),
        "current_industry":    p.get("current_industry"),
    }


def extract_jobs(cid: str, rec: dict) -> list[dict]:
    """One row per job in career_history."""
    rows = []
    for idx, job in enumerate(rec.get("career_history", [])):
        rows.append({
            "candidate_id":   cid,
            "job_index":      idx,
            "company":        job.get("company"),
            "title":          job.get("title"),
            "start_date":     job.get("start_date"),
            "end_date":       job.get("end_date"),
            "duration_months":job.get("duration_months"),
            "is_current":     job.get("is_current"),
            "industry":       job.get("industry"),
            "company_size":   job.get("company_size"),
            "description":    job.get("description"),
        })
    return rows


def extract_skills(cid: str, rec: dict) -> list[dict]:
    """One row per skill entry."""
    rows = []
    for idx, sk in enumerate(rec.get("skills", [])):
        rows.append({
            "candidate_id":   cid,
            "skill_index":    idx,
            "name":           sk.get("name"),
            "proficiency":    sk.get("proficiency"),
            "endorsements":   sk.get("endorsements", 0),
            "duration_months":sk.get("duration_months", 0),
        })
    return rows


def extract_education(cid: str, rec: dict) -> list[dict]:
    """One row per education record."""
    rows = []
    for idx, ed in enumerate(rec.get("education", [])):
        rows.append({
            "candidate_id":  cid,
            "edu_index":     idx,
            "institution":   ed.get("institution"),
            "degree":        ed.get("degree"),
            "field_of_study":ed.get("field_of_study"),
            "start_year":    ed.get("start_year"),
            "end_year":      ed.get("end_year"),
            "grade":         ed.get("grade"),
            "tier":          ed.get("tier"),           # tier_1 / tier_2 / tier_3
        })
    return rows


def extract_redrob(cid: str, rec: dict) -> dict:
    """
    One row per candidate — all 23 Redrob behavioral signals.

    CORRECTIONS vs original Step 1 spec:
      REMOVED (don't exist in data):
        - stackoverflow_reputation
        - kaggle_tier

      ADDED (exist in data, were missing):
        - signup_date
        - profile_views_received_30d
        - applications_submitted_30d
        - connection_count
        - endorsements_received
        - search_appearance_30d

      FIXED:
        - skill_assessment_scores  → stored as JSON string (it's a dict, not a scalar)
        - expected_salary_min/max  → nested under expected_salary_range_inr_lpa

    SENTINEL VALUES (must be preserved, never treated as 0):
        - github_activity_score = -1  → no GitHub linked  (not a negative signal)
        - offer_acceptance_rate = -1  → no offer history  (not 0% acceptance)
    """
    r = rec.get("redrob_signals", {})

    # Salary is nested
    salary = r.get("expected_salary_range_inr_lpa") or {}

    # skill_assessment_scores is a dict — serialise to JSON so parquet stays flat
    assessment_raw = r.get("skill_assessment_scores") or {}
    assessment_json = json.dumps(assessment_raw) if assessment_raw else None

    return {
        "candidate_id":                cid,
        # ── Completeness & activity ─────────────────────────────────────────
        "profile_completeness_score":  r.get("profile_completeness_score"),
        "signup_date":                 r.get("signup_date"),
        "last_active_date":            r.get("last_active_date"),
        "open_to_work_flag":           r.get("open_to_work_flag"),
        # ── Recruiter engagement ────────────────────────────────────────────
        "profile_views_received_30d":  r.get("profile_views_received_30d"),
        "applications_submitted_30d":  r.get("applications_submitted_30d"),
        "recruiter_response_rate":     r.get("recruiter_response_rate"),
        "avg_response_time_hours":     r.get("avg_response_time_hours"),
        # ── Assessment scores (dict → JSON string) ──────────────────────────
        "skill_assessment_scores_json":assessment_json,
        # ── Network & endorsements ──────────────────────────────────────────
        "connection_count":            r.get("connection_count"),
        "endorsements_received":       r.get("endorsements_received"),
        # ── Logistics ───────────────────────────────────────────────────────
        "notice_period_days":          r.get("notice_period_days"),
        "expected_salary_min_lpa":     salary.get("min"),
        "expected_salary_max_lpa":     salary.get("max"),
        "preferred_work_mode":         r.get("preferred_work_mode"),
        "willing_to_relocate":         r.get("willing_to_relocate"),
        # ── External platform scores ─────────────────────────────────────────
        # -1 = not linked / no history — preserve, never coerce to 0
        "github_activity_score":       r.get("github_activity_score"),
        # ── Search visibility ────────────────────────────────────────────────
        "search_appearance_30d":       r.get("search_appearance_30d"),
        "saved_by_recruiters_30d":     r.get("saved_by_recruiters_30d"),
        # ── Conversion signals ───────────────────────────────────────────────
        "interview_completion_rate":   r.get("interview_completion_rate"),
        "offer_acceptance_rate":       r.get("offer_acceptance_rate"),   # -1 = no history
        # ── Verification ─────────────────────────────────────────────────────
        "verified_email":              r.get("verified_email"),
        "verified_phone":              r.get("verified_phone"),
        "linkedin_connected":          r.get("linkedin_connected"),
    }

In [7]:
def parse(input_path: str):
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

    base_rows  = []
    job_rows   = []
    skill_rows = []
    edu_rows   = []
    redrob_rows= []

    total = 0
    errors = 0

    print(f"[Step 1] Reading: {input_path}")

    with open_input(input_path) as fh:
        for line_num, raw_line in enumerate(fh, start=1):
            line = raw_line.strip()
            if not line:
                continue
            try:
                rec = orjson.loads(line)
            except Exception as e:
                errors += 1
                print(f"  [WARN] Line {line_num}: JSON parse error — {e}", file=sys.stderr)
                continue

            cid = rec.get("candidate_id")
            if not cid:
                errors += 1
                print(f"  [WARN] Line {line_num}: missing candidate_id — skipping", file=sys.stderr)
                continue

            base_rows.append(extract_base(cid, rec))
            job_rows.extend(extract_jobs(cid, rec))
            skill_rows.extend(extract_skills(cid, rec))
            edu_rows.extend(extract_education(cid, rec))
            redrob_rows.append(extract_redrob(cid, rec))
            total += 1

            if total % 10_000 == 0:
                print(f"  ... {total:,} candidates parsed")

    print(f"\n[Step 1] Parsed {total:,} candidates ({errors} errors)")

    # ── Write Parquet ────────────────────────────────────────────────────────
    tables = {
        "candidate_base":      pd.DataFrame(base_rows),
        "candidate_jobs":      pd.DataFrame(job_rows),
        "candidate_skills":    pd.DataFrame(skill_rows),
        "candidate_education": pd.DataFrame(edu_rows),
        "candidate_redrob":    pd.DataFrame(redrob_rows),
    }

    for name, df in tables.items():
        path = ARTIFACTS_DIR / f"{name}.parquet"
        df.to_parquet(path, index=False)
        print(f"  Wrote {path}  ({len(df):,} rows, {df.shape[1]} cols)")

    # ── Sanity checks ────────────────────────────────────────────────────────
    print("\n[Step 1] Sanity checks:")
    base_df   = pd.read_parquet(ARTIFACTS_DIR / "candidate_base.parquet")
    redrob_df = pd.read_parquet(ARTIFACTS_DIR / "candidate_redrob.parquet")

    base_unique   = base_df["candidate_id"].nunique()
    redrob_unique = redrob_df["candidate_id"].nunique()

    print(f"  candidate_base   unique IDs: {base_unique:,}  (expected 100,000)")
    print(f"  candidate_redrob unique IDs: {redrob_unique:,}  (expected 100,000)")

    expected = 100_000
    if total == expected:
        print(f"  ✓ All {expected:,} candidates parsed successfully")
    else:
        print(f"  ✗ Expected {expected:,}, got {total:,} — check for errors", file=sys.stderr)

    # Check no candidate lost across joins
    jobs_df   = pd.read_parquet(ARTIFACTS_DIR / "candidate_jobs.parquet")
    skills_df = pd.read_parquet(ARTIFACTS_DIR / "candidate_skills.parquet")
    jobs_ids_valid   = jobs_df["candidate_id"].isin(base_df["candidate_id"]).all()
    skills_ids_valid = skills_df["candidate_id"].isin(base_df["candidate_id"]).all()
    print(f"  candidate_jobs   all IDs valid: {jobs_ids_valid}")
    print(f"  candidate_skills all IDs valid: {skills_ids_valid}")

    print("\n[Step 1] Done. Artifacts written to:", ARTIFACTS_DIR.resolve())
    return tables

In [12]:
def verify():
    """Quick load and shape check on existing artifacts."""
    print("[Verify] Loading artifacts...")

    dfs = {
        name: pd.read_parquet(ARTIFACTS_DIR / f"{name}.parquet")
        for name in [
            "candidate_base", "candidate_jobs",
            "candidate_skills", "candidate_education", "candidate_redrob"
        ]
    }

    for name, df in dfs.items():
        print(f"  {name:30s}  shape={str(df.shape):20s}  columns={list(df.columns)}")

    base   = dfs["candidate_base"]
    redrob = dfs["candidate_redrob"]

    assert base["candidate_id"].nunique() == 100_000,   "base: expected 100k unique IDs"
    assert redrob["candidate_id"].nunique() == 100_000, "redrob: expected 100k unique IDs"

    print("\n[Verify] ✓ All assertions passed")

In [9]:
%%time
tables = parse(DATASET_PATH)

[Step 1] Reading: E:\Coding\Projects\redrob-ranker\dataset\candidates.jsonl
  ... 10,000 candidates parsed
  ... 20,000 candidates parsed
  ... 30,000 candidates parsed
  ... 40,000 candidates parsed
  ... 50,000 candidates parsed
  ... 60,000 candidates parsed
  ... 70,000 candidates parsed
  ... 80,000 candidates parsed
  ... 90,000 candidates parsed
  ... 100,000 candidates parsed

[Step 1] Parsed 100,000 candidates (0 errors)
  Wrote E:\Coding\Projects\redrob-ranker\dataset\artifacts\candidate_base.parquet  (100,000 rows, 11 cols)
  Wrote E:\Coding\Projects\redrob-ranker\dataset\artifacts\candidate_jobs.parquet  (300,171 rows, 11 cols)
  Wrote E:\Coding\Projects\redrob-ranker\dataset\artifacts\candidate_skills.parquet  (960,302 rows, 6 cols)
  Wrote E:\Coding\Projects\redrob-ranker\dataset\artifacts\candidate_education.parquet  (139,778 rows, 9 cols)
  Wrote E:\Coding\Projects\redrob-ranker\dataset\artifacts\candidate_redrob.parquet  (100,000 rows, 25 cols)

[Step 1] Sanity checks:

In [13]:
%%time
verify()

[Verify] Loading artifacts...
  candidate_base                  shape=(100000, 11)          columns=['candidate_id', 'anonymized_name', 'headline', 'summary', 'location', 'country', 'years_of_experience', 'current_title', 'current_company', 'current_company_size', 'current_industry']
  candidate_jobs                  shape=(300171, 11)          columns=['candidate_id', 'job_index', 'company', 'title', 'start_date', 'end_date', 'duration_months', 'is_current', 'industry', 'company_size', 'description']
  candidate_skills                shape=(960302, 6)           columns=['candidate_id', 'skill_index', 'name', 'proficiency', 'endorsements', 'duration_months']
  candidate_education             shape=(139778, 9)           columns=['candidate_id', 'edu_index', 'institution', 'degree', 'field_of_study', 'start_year', 'end_year', 'grade', 'tier']
  candidate_redrob                shape=(100000, 25)          columns=['candidate_id', 'profile_completeness_score', 'signup_date', 'last_active_da

In [15]:
type(tables)

dict

In [16]:
tables.keys()

dict_keys(['candidate_base', 'candidate_jobs', 'candidate_skills', 'candidate_education', 'candidate_redrob'])

In [18]:
tables['candidate_base'].head()

,candidate_id,anonymized_name,headline,summary,location,country,years_of_experience,current_title,current_company,current_company_size,current_industry
0,CAND_0000001,Ira Vora,"Backend Engineer | SQL, Spark, Cloud",Software / data professional with 6.9 years of...,Toronto,Canada,6.9,Backend Engineer,Mindtree,10001+,IT Services
1,CAND_0000002,Saanvi Sethi,Operations Manager | 12.5+ yrs experience,Professional with 12.5+ years of experience. M...,"Chennai, Tamil Nadu",India,12.5,Operations Manager,Wipro,10001+,IT Services
2,CAND_0000003,Yash Agarwal,Customer Support | 1.1+ yrs experience,Professional with 1.1+ years of experience. I'...,Austin,USA,1.1,Customer Support,TCS,10001+,IT Services
3,CAND_0000004,Anil Bose,Marketing Manager | Driving business outcomes,Professional with 3.8+ years of experience. My...,Sydney,Australia,3.8,Marketing Manager,Dunder Mifflin,201-500,Paper Products
4,CAND_0000005,Aisha Sethi,Accountant | Helping teams scale,Professional with 11.0+ years of experience. I...,"Gurgaon, Haryana",India,11.0,Accountant,Stark Industries,1001-5000,Manufacturing
